[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/07_logistic_regression/02_from_scratch_and_optimization.ipynb)

# Bài 02 — Logistic Regression from scratch: Gradient Descent, BFGS, scikit-learn

**🎯 Mục tiêu**
- Cài đặt cost, gradient, gradient descent cho Logistic Regression bằng NumPy; kiểm tra bằng **gradient check**.
- Tối ưu bằng `scipy.optimize.minimize` (BFGS — "advanced optimization" của bài 01) và so sánh.
- Đóng gói thành class theo API scikit-learn; đối chiếu với `sklearn.linear_model.LogisticRegression` (và hiểu tham số `C`).

Dữ liệu: dự đoán khách hàng **churn** (`customer_churn.csv`) từ vài feature số.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/07_logistic_regression

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split

from ml_course.utils import numerical_gradient, relative_error, sigmoid
from ml_course.visualization import set_style

set_style()

df = pd.read_csv("../../data/sample/customer_churn.csv")
features = ["tenure_months", "monthly_charges", "support_calls", "n_products"]
X_raw = df[features].to_numpy(float)
y = df["churn"].to_numpy(float)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y, test_size=0.25, random_state=0, stratify=y)
mu, sd = X_train_raw.mean(0), X_train_raw.std(0)
X_train = np.c_[np.ones(len(X_train_raw)), (X_train_raw - mu) / sd]
X_test = np.c_[np.ones(len(X_test_raw)), (X_test_raw - mu) / sd]
print(X_train.shape, "tỉ lệ churn:", y_train.mean().round(3))

## 1. Cost và gradient

$$h = \sigma(X\theta), \qquad J(\theta) = -\frac1m\left[y^\top\log h + (1 - y)^\top\log(1 - h)\right], \qquad \nabla J(\theta) = \frac1m X^\top(h - y)$$

Hai chi tiết số học quan trọng:
- Dùng **sigmoid ổn định số học** (`ml_course.utils.sigmoid`) — tránh `exp` tràn số với $z$ rất âm.
- `clip` xác suất vào $[\varepsilon, 1 - \varepsilon]$ trước khi lấy log — tránh `log(0)`.

In [ ]:
def cost(theta, X, y, eps=1e-15):
    h = np.clip(sigmoid(X @ theta), eps, 1 - eps)
    return -np.mean(y * np.log(h) + (1 - y) * np.log(1 - h))


def gradient(theta, X, y):
    return X.T @ (sigmoid(X @ theta) - y) / len(y)


theta0 = np.zeros(X_train.shape[1])
print(f"J(θ = 0) = {cost(theta0, X_train, y_train):.4f}  (= ln 2 = {np.log(2):.4f}, vì mọi dự đoán đều là 0.5)")

### Gradient check

Luôn kiểm tra gradient tự viết bằng sai phân số trước khi dùng (chương 02, bài 04):

In [ ]:
theta_rand = np.random.default_rng(0).normal(size=X_train.shape[1])
num = numerical_gradient(lambda t: cost(t, X_train, y_train), theta_rand.copy())
print("relative error:", f"{relative_error(gradient(theta_rand, X_train, y_train), num):.2e}", "→ < 1e-7: gradient đúng")

## 2. Gradient Descent

In [ ]:
def gradient_descent(X, y, alpha=0.5, num_iters=500):
    theta = np.zeros(X.shape[1])
    history = []
    for _ in range(num_iters):
        theta -= alpha * gradient(theta, X, y)
        history.append(cost(theta, X, y))
    return theta, history


start = time.perf_counter()
theta_gd, J_hist = gradient_descent(X_train, y_train)
t_gd = time.perf_counter() - start
plt.plot(J_hist)
plt.xlabel("vòng lặp")
plt.ylabel("J(θ)")
plt.title("Cross-entropy giảm dần")
plt.show()
print("θ (GD) =", theta_gd.round(4), f"| {t_gd * 1000:.0f} ms")

## 3. Advanced optimization: `scipy.optimize.minimize`

Truyền vào hàm cost và gradient (`jac`) — thuật toán **BFGS** tự chọn bước đi, hội tụ sau vài chục vòng lặp:

In [ ]:
start = time.perf_counter()
res = minimize(cost, theta0, args=(X_train, y_train), jac=gradient, method="BFGS")
t_bfgs = time.perf_counter() - start
print(f"BFGS: hội tụ = {res.success}, {res.nit} vòng lặp, {t_bfgs * 1000:.0f} ms, J = {res.fun:.6f}")
print("θ (BFGS) =", res.x.round(4))
print(f"GD sau 500 vòng: J = {J_hist[-1]:.6f}")

BFGS dùng thông tin **độ cong** (xấp xỉ Hessian) để chọn hướng và độ dài bước tốt hơn GD → ít vòng lặp hơn rất nhiều, không cần learning rate.

### Newton's method / IRLS *(nâng cao)*

Với Logistic Regression, Hessian có công thức đóng $H = \frac1m X^\top S X$, $S = \operatorname{diag}(h_i(1 - h_i))$. Phương pháp Newton $\theta \leftarrow \theta - H^{-1}\nabla J$ hội tụ trong khoảng 5–10 bước (thuật toán kinh điển trong thống kê — *Iteratively Reweighted Least Squares*):

In [ ]:
theta = np.zeros(X_train.shape[1])
for it in range(8):
    h = sigmoid(X_train @ theta)
    H = X_train.T @ (X_train * (h * (1 - h))[:, None]) / len(y_train)
    theta -= np.linalg.solve(H, gradient(theta, X_train, y_train))
    print(f"Newton bước {it + 1}: J = {cost(theta, X_train, y_train):.8f}")

## 4. Đóng gói thành class và đánh giá

Phiên bản đầy đủ nằm trong `src/ml_course/algorithms/linear_models.py` (`LogisticRegressionScratch`). Đánh giá trên tập test:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

from ml_course.algorithms import LogisticRegressionScratch

ours = LogisticRegressionScratch(lr=0.5, n_iters=3000).fit(X_train[:, 1:], y_train)
sk = LogisticRegression(C=1e6).fit(X_train[:, 1:], y_train)        # C rất lớn ≈ không regularization
for name, model in [("from scratch", ours), ("scikit-learn", sk)]:
    p = model.predict_proba(X_test[:, 1:])[:, 1]
    print(f"{name:13s} accuracy = {accuracy_score(y_test, p >= 0.5):.3f} | log-loss = {log_loss(y_test, p):.4f} | ROC-AUC = {roc_auc_score(y_test, p):.3f}")
print("hệ số gần như trùng nhau:", np.allclose(ours.coef_, sk.coef_.ravel(), atol=1e-2), "| scratch:", ours.coef_.round(3), "| sklearn:", sk.coef_.ravel().round(3))

### Tham số `C` của scikit-learn

`sklearn.linear_model.LogisticRegression` **mặc định có regularization L2** với `C = 1.0`, trong đó $C = 1/\lambda$ — **C càng nhỏ, regularization càng mạnh** (ngược với `alpha` của Ridge!). Loss của sklearn:

$$\min_\theta\; \frac12\lVert\theta\rVert^2 + C\sum_{i=1}^m \text{logloss}_i$$

> ⚠️ Lỗi phổ biến: quên rằng sklearn đã regularize mặc định → hệ số "bị co" khác với kết quả thống kê; với feature chưa chuẩn hoá, regularization phạt không công bằng. Luôn chuẩn hoá feature khi dùng `LogisticRegression`.

In [ ]:
for C in [0.001, 0.01, 1.0, 1e6]:
    m = LogisticRegression(C=C).fit(X_train[:, 1:], y_train)
    print(f"C = {C:<8g} ‖θ‖ = {np.linalg.norm(m.coef_):.3f} | log-loss test = {log_loss(y_test, m.predict_proba(X_test[:, 1:])[:, 1]):.4f}")

## 5. Diễn giải hệ số bằng odds ratio

Feature đã chuẩn hoá → hệ số = thay đổi log-odds khi feature tăng **1 độ lệch chuẩn**; $e^{\theta_j}$ là **odds ratio**:

In [ ]:
coef = pd.DataFrame({"hệ số (log-odds / 1 std)": sk.coef_.ravel(), "odds ratio": np.exp(sk.coef_.ravel()),
                     "1 std bằng": sd}, index=features).round(3)
coef

Đọc: mỗi **1 độ lệch chuẩn** số cuộc gọi hỗ trợ (~1.3 cuộc) làm **odds** churn tăng gấp `odds ratio` lần; tenure càng dài odds churn càng giảm. (Nhớ: đây là liên hệ có điều kiện, không phải nhân quả.)

## 🧠 Kiểm tra nhanh

1. Vì sao $J(\theta = 0) = \ln 2$ với mọi dữ liệu nhị phân?
2. Gradient check cho relative error = 0.3. Bạn làm gì tiếp theo?
3. `LogisticRegression(C=0.01)` và `LogisticRegression(C=100)` — cái nào regularize mạnh hơn?
4. Odds ratio = 0.5 cho feature `tenure` nghĩa là gì?

## 📚 Tài liệu tham khảo

- Nocedal & Wright — *Numerical Optimization* (BFGS, Newton)
- scikit-learn User Guide — Logistic regression (solvers, regularization)
- ISLR, §4.3